In [ ]:
# ============================================================
# TASK 03 - INTERACTIVE HR EMPLOYEE ATTRITION DASHBOARD
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import os
import glob
import json

from google.colab import files

# ============================================================
# 1. UPLOAD DATASET
# ============================================================

print("Upload your IBM HR Attrition ZIP or CSV file")

uploaded = files.upload()

uploaded_file = list(uploaded.keys())[0]

# ============================================================
# 2. READ DATASET
# ============================================================

if uploaded_file.lower().endswith(".zip"):

    extract_folder = "hr_dataset"

    os.makedirs(extract_folder, exist_ok=True)

    with zipfile.ZipFile(uploaded_file, "r") as zip_ref:
        zip_ref.extractall(extract_folder)

    csv_files = glob.glob(
        extract_folder + "/**/*.csv",
        recursive=True
    )

    if len(csv_files) == 0:
        raise FileNotFoundError(
            "No CSV file found inside the ZIP file."
        )

    file_name = csv_files[0]

else:
    file_name = uploaded_file


df = pd.read_csv(file_name)

print("\n======================================")
print("DATASET LOADED SUCCESSFULLY")
print("======================================")

print("File:", file_name)
print("Rows:", len(df))
print("Columns:", len(df.columns))


# ============================================================
# 3. CLEAN DATA
# ============================================================

df = df.drop_duplicates()

# Convert important numeric columns
numeric_columns = [
    "Age",
    "MonthlyIncome",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "DistanceFromHome",
    "JobSatisfaction",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "WorkLifeBalance"
]

for col in numeric_columns:

    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# ============================================================
# 4. CREATE AGE GROUP
# ============================================================

df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0, 24, 34, 44, 54, 100],
    labels=[
        "Under 25",
        "25-34",
        "35-44",
        "45-54",
        "55+"
    ]
)


# ============================================================
# 5. BASIC STATISTICS
# ============================================================

total_employees = len(df)

employees_left = (
    df["Attrition"]
    .eq("Yes")
    .sum()
)

employees_stayed = (
    df["Attrition"]
    .eq("No")
    .sum()
)

attrition_rate = (
    employees_left / total_employees * 100
)

print("\n========== HR SUMMARY ==========")
print("Total Employees :", total_employees)
print("Employees Left   :", employees_left)
print("Employees Stayed :", employees_stayed)
print("Attrition Rate   :", round(attrition_rate, 2), "%")


# ============================================================
# 6. CREATE HTML DASHBOARD
# ============================================================

# Convert dataframe to JSON
data_json = df.to_json(
    orient="records"
)

html = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>Employee Attrition Dashboard</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

body {{
    font-family: Arial, sans-serif;
    background: #f4f6f8;
    margin: 0;
    padding: 25px;
}}

h1 {{
    text-align: center;
    color: #1f2937;
}}

.dashboard {{
    max-width: 1400px;
    margin: auto;
}}

.filters {{
    background: white;
    padding: 20px;
    border-radius: 12px;
    margin-bottom: 20px;
    display: flex;
    gap: 15px;
    flex-wrap: wrap;
}}

select {{
    padding: 10px;
    border-radius: 6px;
    border: 1px solid #ccc;
    min-width: 180px;
}}

.cards {{
    display: grid;
    grid-template-columns:
        repeat(4, 1fr);
    gap: 15px;
    margin-bottom: 20px;
}}

.card {{
    background: white;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
}}

.card h2 {{
    margin: 0;
    font-size: 30px;
}}

.card p {{
    color: #666;
}}

.charts {{
    display: grid;
    grid-template-columns:
        repeat(2, 1fr);
    gap: 20px;
}}

.chart {{
    background: white;
    border-radius: 12px;
    padding: 10px;
}}

.insights {{
    background: white;
    margin-top: 20px;
    padding: 20px;
    border-radius: 12px;
}}

@media(max-width:900px) {{

    .cards {{
        grid-template-columns:
            repeat(2, 1fr);
    }}

    .charts {{
        grid-template-columns:
            1fr;
    }}

}}

</style>

</head>


<body>

<div class="dashboard">

<h1>
EMPLOYEE ATTRITION ANALYSIS DASHBOARD
</h1>


<!-- FILTERS -->

<div class="filters">

<label>
Department
<br>

<select id="department"
onchange="updateDashboard()">

<option value="All">
All Departments
</option>

</select>

</label>


<label>
Age Group
<br>

<select id="agegroup"
onchange="updateDashboard()">

<option value="All">
All Age Groups
</option>

</select>

</label>


<label>
Job Role
<br>

<select id="jobrole"
onchange="updateDashboard()">

<option value="All">
All Job Roles
</option>

</select>

</label>


<label>
Overtime
<br>

<select id="overtime"
onchange="updateDashboard()">

<option value="All">
All
</option>

</select>

</label>

</div>


<!-- KPI CARDS -->

<div class="cards">

<div class="card">

<h2 id="totalEmployees">
0
</h2>

<p>
TOTAL EMPLOYEES
</p>

</div>


<div class="card">

<h2 id="employeesLeft">
0
</h2>

<p>
EMPLOYEES LEFT
</p>

</div>


<div class="card">

<h2 id="employeesStayed">
0
</h2>

<p>
EMPLOYEES STAYED
</p>

</div>


<div class="card">

<h2 id="attritionRate">
0%
</h2>

<p>
ATTRITION RATE
</p>

</div>

</div>


<!-- CHARTS -->

<div class="charts">

<div class="chart">

<div id="departmentChart"></div>

</div>


<div class="chart">

<div id="ageChart"></div>

</div>


<div class="chart">

<div id="jobChart"></div>

</div>


<div class="chart">

<div id="overtimeChart"></div>

</div>

</div>


<!-- INSIGHTS -->

<div class="insights">

<h2>
Why Are Employees Leaving?
</h2>

<p id="insightText">
Select filters to analyze employee attrition.
</p>

</div>


</div>


<script>

const rawData = {data_json};


let data = rawData.map(row => {{

    return row;

}});


// ==========================================================
// GET UNIQUE VALUES
// ==========================================================

function uniqueValues(column) {{

    return [
        ...new Set(
            data.map(
                row => row[column]
            )
        )
    ]
    .filter(
        value =>
        value !== null &&
        value !== undefined &&
        value !== ""
    )
    .sort();

}}


// ==========================================================
// POPULATE FILTERS
// ==========================================================

function populateFilter(
    elementId,
    column
) {{

    const select =
        document.getElementById(
            elementId
        );

    uniqueValues(column)
    .forEach(value => {{

        const option =
            document.createElement(
                "option"
            );

        option.value = value;

        option.textContent = value;

        select.appendChild(
            option
        );

    }});

}}


populateFilter(
    "department",
    "Department"
);

populateFilter(
    "agegroup",
    "Age Group"
);

populateFilter(
    "jobrole",
    "JobRole"
);

populateFilter(
    "overtime",
    "OverTime"
);


// ==========================================================
// FILTER DATA
// ==========================================================

function getFilteredData() {{

    const department =
        document.getElementById(
            "department"
        ).value;

    const agegroup =
        document.getElementById(
            "agegroup"
        ).value;

    const jobrole =
        document.getElementById(
            "jobrole"
        ).value;

    const overtime =
        document.getElementById(
            "overtime"
        ).value;


    return data.filter(row => {{

        return (

            (
                department === "All" ||
                row.Department === department
            )

            &&

            (
                agegroup === "All" ||
                row["Age Group"] === agegroup
            )

            &&

            (
                jobrole === "All" ||
                row.JobRole === jobrole
            )

            &&

            (
                overtime === "All" ||
                row.OverTime === overtime
            )

        );

    }});

}}


// ==========================================================
// COUNT FUNCTION
// ==========================================================

function countBy(
    dataset,
    column,
    filterAttrition = false
) {{

    const result = {{}};

    dataset.forEach(row => {{

        if (
            filterAttrition &&
            row.Attrition !== "Yes"
        ) {{
            return;
        }}

        const key = row[column];

        if (
            key === null ||
            key === undefined
        ) {{
            return;
        }}

        result[key] =
            (result[key] || 0) + 1;

    }});

    return result;

}}


// ==========================================================
// UPDATE DASHBOARD
// ==========================================================

function updateDashboard() {{

    const filtered =
        getFilteredData();


    // ------------------------------------------------------
    // KPI
    // ------------------------------------------------------

    const total =
        filtered.length;


    const left =
        filtered.filter(
            row =>
            row.Attrition === "Yes"
        ).length;


    const stayed =
        filtered.filter(
            row =>
            row.Attrition === "No"
        ).length;


    const rate =
        total > 0
        ? (left / total * 100)
        : 0;


    document.getElementById(
        "totalEmployees"
    ).innerText = total;


    document.getElementById(
        "employeesLeft"
    ).innerText = left;


    document.getElementById(
        "employeesStayed"
    ).innerText = stayed;


    document.getElementById(
        "attritionRate"
    ).innerText =
        rate.toFixed(2) + "%";


    // ------------------------------------------------------
    // Department Chart
    // ------------------------------------------------------

    const departmentData =
        countBy(
            filtered,
            "Department",
            true
        );


    Plotly.newPlot(
        "departmentChart",
        [{{
            x: Object.keys(
                departmentData
            ),

            y: Object.values(
                departmentData
            ),

            type: "bar"
        }}],
        {{
            title:
            "Attrition by Department",

            xaxis:
            {{
                title: "Department"
            }},

            yaxis:
            {{
                title:
                "Employees Left"
            }},

            margin:
            {{
                t: 60
            }}
        }}
    );


    // ------------------------------------------------------
    // Age Group Chart
    // ------------------------------------------------------

    const ageData =
        countBy(
            filtered,
            "Age Group",
            true
        );


    const ageOrder = [
        "Under 25",
        "25-34",
        "35-44",
        "45-54",
        "55+"
    ];


    Plotly.newPlot(
        "ageChart",
        [{{
            x: ageOrder,

            y: ageOrder.map(
                x =>
                ageData[x] || 0
            ),

            type: "bar"
        }}],
        {{
            title:
            "Attrition by Age Group",

            xaxis:
            {{
                title:
                "Age Group"
            }},

            yaxis:
            {{
                title:
                "Employees Left"
            }},

            margin:
            {{
                t: 60
            }}
        }}
    );


    // ------------------------------------------------------
    // Job Role Chart
    // ------------------------------------------------------

    const jobData =
        countBy(
            filtered,
            "JobRole",
            true
        );


    Plotly.newPlot(
        "jobChart",
        [{{
            x: Object.values(
                jobData
            ),

            y: Object.keys(
                jobData
            ),

            type: "bar",
            orientation: "h"
        }}],
        {{
            title:
            "Attrition by Job Role",

            xaxis:
            {{
                title:
                "Employees Left"
            }},

            yaxis:
            {{
                title:
                "Job Role"
            }},

            margin:
            {{
                t: 60
            }}
        }}
    );


    // ------------------------------------------------------
    // Overtime Chart
    // ------------------------------------------------------

    const overtimeData =
        countBy(
            filtered,
            "OverTime",
            true
        );


    Plotly.newPlot(
        "overtimeChart",
        [{{
            x: Object.keys(
                overtimeData
            ),

            y: Object.values(
                overtimeData
            ),

            type: "bar"
        }}],
        {{
            title:
            "Attrition by Overtime",

            xaxis:
            {{
                title:
                "Overtime"
            }},

            yaxis:
            {{
                title:
                "Employees Left"
            }},

            margin:
            {{
                t: 60
            }}
        }}
    );


    // ------------------------------------------------------
    // INSIGHT
    // ------------------------------------------------------

    let insight = "";

    if (total === 0) {{

        insight =
        "No employees match the selected filters.";

    }} else {{

        const overtimeYes =
            filtered.filter(
                row =>
                row.Attrition === "Yes"
                &&
                row.OverTime === "Yes"
            ).length;


        const overtimeNo =
            filtered.filter(
                row =>
                row.Attrition === "Yes"
                &&
                row.OverTime === "No"
            ).length;


        if (
            overtimeYes >
            overtimeNo
        ) {{

            insight =
            "Overtime is associated with a higher number of employees leaving in the selected data.";

        }} else {{

            insight =
            "The selected filters show that factors other than overtime may be more prominent for this group.";

        }}

    }}


    document.getElementById(
        "insightText"
    ).innerText = insight;

}}


// ==========================================================
// INITIAL DASHBOARD
// ==========================================================

updateDashboard();

</script>

</body>

</html>
"""


# ============================================================
# 7. SAVE HTML
# ============================================================

output_file = (
    "HR_Employee_Attrition_Dashboard.html"
)

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(html)


print("\n======================================")
print("INTERACTIVE DASHBOARD CREATED!")
print("======================================")

print(
    "Output:",
    output_file
)

print(
    "\nOpen the HTML file in Chrome/Edge."
)


# ============================================================
# 8. DOWNLOAD DASHBOARD
# ============================================================

files.download(
    output_file
)

Upload your IBM HR Attrition ZIP or CSV file


Saving archive (3).zip to archive (3).zip

DATASET LOADED SUCCESSFULLY
File: hr_dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv
Rows: 1470
Columns: 35

========== HR SUMMARY ==========
Total Employees : 1470
Employees Left   : 237
Employees Stayed : 1233
Attrition Rate   : 16.12 %

INTERACTIVE DASHBOARD CREATED!
Output: HR_Employee_Attrition_Dashboard.html

Open the HTML file in Chrome/Edge.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>